<a href="https://colab.research.google.com/github/3D-fossils-Haag/vert-nsm/blob/main/demos/shape_completion_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Squamate Vertebra Shape Completion Demo**   
*Last edited 22 Oct 2026*


This notebook demonstrates shape completion for partial vertebrae (modern and fossil) using a trained Neural Shape Model (NSM; Gatti et al. 2025, Park et al. 2019). It can be run fully in demo mode without connecting to your Google Drive. Adjust parameters using form fields and make sure your runtime environment is set to run on GPU.

Modern vertebra meshes are derived from micro-CT data produced by the oVert Initiative (Blackburn et al. 2024). Fossil vertebra were downloaded from MorphoSource ([UF546657](https://doi.org/10.17602/M2/M600663); [UF271967](https://n2t.net/ark:/87602/m4/M69199)). All vertebrae were aligned and scaled using [MorphoWeave](https://github.com/agporto/SlicerMorphoWeave) before training.


**References**
* Blackburn et al. 2024, BioScience. https://doi.org/10.1093/biosci/biad120
* Gatti et al. 2025, IEEE TMI. https://doi.org/10.1109/tmi.2024.3485613
* Park et al. 2019, CVPR. https://doi.org/10.48550/arXiv.1901.05103
* Porto et al. 2026, bioRxiv. https://doi.org/10.64898/2026.09.28.755222

# 1. Installs & Imports
---
This notebook can run fully in the cloud or can be connected to your Google Drive to save results. Run these code blocks to set up your environment before proceeding with shape completion.

## Check that your Colab runtime environment is set to use GPU.
Go to the top right corner of this notebook → Click the arrow → Change runtime type  → Hardware Accelerator  → any option with GPU

In [ ]:
# Check GPU and CUDA info - make sure Colab Runtime set to GPU
from psutil import virtual_memory

# Check GPU and CUDA
!nvcc --version
gpu = !nvidia-smi
gpu = '\n'.join(gpu)
print('\033[91mNot connected to a GPU\033[0m' if 'failed' in gpu else gpu)

# Check RAM
ram = virtual_memory().total / 1e9
print(f'\033[92mYour runtime has {ram:.1f} GB of RAM\033[0m\n')

## Choose where to save results
This notebook can be run fully within the Colab runtime environment (files are deleted after each session) or it can be connected to your Google Drive. To customize, run the form fields on the right to set the parameters for "save" and "base_wd" to determine your base working directory where files will go.

In [ ]:
# Choose where to save results

# Use dropdown menu on right
save = "in Colab runtime (files deleted after each session)" #@param ["in my Google Drive", "in Colab runtime (files deleted after each session)"]

# Mount google drive to export image tagging file(s)
if 'Google Drive' in save:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Type in the path to your project wd in form field on right
base_wd = "/content/drive/MyDrive" # @param ["/content/drive/MyDrive/nsm"] {"allow-input":true}
wd = base_wd + "/nsm"
print(f"\033[92mWorking directory set to: \n{wd}\033[0m")

## Set up environment and install NSM
Machine learning environments have specific versions required for certain libaries. The code below first ensures that all the versions installed are compatible with NSM. Then, it clones NSM from GitHub, installs other required packages, and installs NSM.

In [ ]:
# Set up environment and install NSM

# Define ipywidget version to avoid need for runtime restart
!pip install -q ipywidgets==8.1.8 widgetsnbextension==4.0.15
import importlib
import ipywidgets
importlib.reload(ipywidgets)

# Other imports
import os
import sys

# Install PyTorch with CUDA support (Colab typically has CUDA 11.8 or 12.x)
print("\033[92mSetting up environment...\033[0m")
print("\n\033[33m-----This will take a few minutes----\033[0m")
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

# Install other dependencies
!pip install pyvista mskt open3d scikit-learn matplotlib pandas numpy scipy
!pip install nibabel scikit-image opencv-python open3d

# Clone NSM repository
if not os.path.exists(wd):
    print("Cloning NSM repository...")
    os.makedirs(base_wd, exist_ok=True)
    %cd $base_wd
    !git clone https://github.com/3D-fossils-Haag/nsm.git
else:
    print("NSM directory already exists")

# Navigate to nsm directory and install
%cd $wd

# Install requirements
print("\n-----Installing requirements-----")
!python -m pip install -r requirements.txt

# Install NSM package
print("\n-----Installing NSM-----")
!pip install .

# Add to Python path
sys.path.insert(0, wd)
%cd $wd
print(f"\n\033[92mCurrent working directory set to: {os.getcwd()}\033[0m")

## Import libraries and define functions
Import required libraries and define functions to be used downstream

In [ ]:
# Import libraries and define functions

# For rendering meshes
import pyvista as pv
import plotly.graph_objects as go
import pymskt.mesh.meshes as meshes

# For working with ML
import torch
from NSM.helper_funcs import load_config, load_model_and_latents
from NSM.helper_funcs import NumpyTransform, convert_ply_to_vtk
from NSM.optimization import (sample_near_surface, get_top_k_pcs,
    downsample_partial_pointcloud, optimize_latent_partial,
    normalize_mesh, get_norm_params, encode_latent, encode_latent_pointnet,
                              reconstruct_mesh_from_latent, build_sdf_dataset)
# For working with data
import numpy as np
import random
import json

# Silence future warnings to improve output readability
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*algorithm.*extract_surface.*")

# Plot pyvista mesh interactively using plotly
def pv_to_plotly(mesh, color="deepskyblue", opacity=1.0, scalars=None, colorscale="Viridis"):
    mesh = mesh.extract_surface().triangulate()
    faces = mesh.faces.reshape(-1, 4)
    base = dict(
        x=mesh.points[:, 0], y=mesh.points[:, 1], z=mesh.points[:, 2],
        i=faces[:, 1], j=faces[:, 2], k=faces[:, 3],
        opacity=opacity, flatshading=False,
        lighting=dict(ambient=0.12, diffuse=0.88, specular=0.05,
                      roughness=0.9, fresnel=0.0),
        lightposition=dict(x=0, y=0, z=2))
    if scalars is not None:
        base.update(dict(intensity=scalars, colorscale=colorscale, showscale=True))
    else:
        base.update(dict(color=color))
    return go.Mesh3d(**base)

# Plot pyvista pointcloud interactively using plotly
def pv_points_to_plotly(mesh, color='red', size=4):
    pts = mesh.points
    return go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                        mode='markers',
                        marker=dict(size=size, color=color, opacity=0.3))

# Monkey patch for data types ----
from NSM.helper_funcs import safe_load_mesh_scalars, fixed_point_coords
meshes.Mesh.load_mesh_scalars = safe_load_mesh_scalars
meshes.Mesh.point_coords = property(fixed_point_coords)

import pymskt.mesh.meshTools as meshTools
_original_signed_distance_to_mesh = meshTools.pcu.signed_distance_to_mesh

def _signed_distance_to_mesh_patch(pts, points, faces):
    pts = np.asarray(pts, dtype=np.float64)     # force double precision
    points = np.asarray(points, dtype=np.float64)
    faces = np.asarray(faces, dtype=np.int32)   # ensure integer type for faces
    return _original_signed_distance_to_mesh(pts, points, faces)
meshTools.pcu.signed_distance_to_mesh = _signed_distance_to_mesh_patch
# End monkey patch ----

# 2. Load in models and data
---
Load models and data to do demo shape completion in your Colab runtime environment.

## Download models and meshes to appropriate folders
Use gdown syntax to download demo model and data files to their respective directories. To customize, adjust paths using form fields on right.

In [ ]:
# Download models and meshes to appropriate folders

# Update these paths to point to your model and data
MODEL_DIR = "run_v44" # @param ["run_v44"] {"allow-input":true}
!gdown 1QVmApul8KJsddH8yNjdGmsmR235uwxxt
!unzip -o -q {MODEL_DIR}.zip && rm -f {MODEL_DIR}.zip

# Checkpoint to use
CKPT = "3000" # @param ["3000"] {"allow-input":true}
CKPT_fn = CKPT + '.pth'

# Fossil directory
fossil_dir = "fossils" # @param ["fossils"] {"allow-input":true}
!gdown 1QRQapNyZb3911O9bl-tVqX49TA56LAE7
!unzip -o -q {fossil_dir}.zip && rm -f {fossil_dir}.zip

# Modern vertebrae directory
vertebrae_dir = "vertebrae_meshes" # @param ["vertebrae_meshes"] {"allow-input":true}
%cd $wd
!rm -rf $vertebrae_dir # Delete demo vertebrae_meshes dir from nsm github
!gdown 1EaQJEfryoziFjdfYmI2-UPoF0wvhdnhS
!unzip -o {vertebrae_dir}.zip -d {vertebrae_dir} && rm -f {vertebrae_dir}.zip

# Output directory
OUTPUT_DIR = "shape_completion/predictions" # @param ["outputs"] {"allow-input":true}
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n\033[92mSet up working directory and downloaded model and mesh files.")
print(f"Model directory: {MODEL_DIR}")
print(f"Checkpoint: {CKPT}")
print(f"Output directory: {OUTPUT_DIR}\033[0m")

# 3. Prepare models and data for inference
---

Prepare models and data to run inference (complete the shape from a partial mesh (modern or fossil)).


## Load model and latent codes
Load files that were downloaded in **2. Load models and data**.

In [ ]:
# Load model and latent codes

# Change to model directory
%cd $MODEL_DIR

# Load config
config = load_config(config_path='model_params_config.json')
device = config.get("device", "cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths to model and latent codes
LC_PATH = f'latent_codes/{CKPT}.pth'
MODEL_PATH = f'model/{CKPT}.pth'

# Load model and latents
print("Loading model and latents...")
model, latent_ckpt, latent_codes = load_model_and_latents(MODEL_PATH, LC_PATH, config, device)

# Compute statistics
mean_latent = latent_codes.mean(dim=0, keepdim=True)
latent_std = latent_codes.std().mean()
_, top_k_reg = get_top_k_pcs(latent_codes, threshold=0.99)

# Return to original directory
%cd $wd

print(f"\nLatent size: {config['latent_size']}")
print(f"Number of training samples: {len(latent_codes)}")

## Load mesh
Pick whether to load a modern (vertebrae_dir) or fossil (fossil_dir) vertebrae from the demo directories, then randomly select a file from the directory for inference.

In [ ]:
# Load mesh into latent space

# Pick a mesh
mesh_dir = fossil_dir # @param ["fossil_dir","vertebrae_dir"] {"type":"raw","allow-input":true}
mesh_path = "uf546657.ply" # @param ["uf546657.ply","uf546657_smooth_hollow.vtk"] {"allow-input":true}
print(f"Mesh being loaded from directory: {mesh_dir}\n{mesh_path}\n")

# Setup output directory
mesh_name = os.path.splitext(os.path.basename(mesh_path))[0]
outfpath = os.path.join(OUTPUT_DIR, mesh_name)
os.makedirs(outfpath, exist_ok=True)
print(f"Saving results to output directory: {outfpath}")

# Convert PLY to VTK if needed
mesh_path = os.path.join(mesh_dir, mesh_path)
vert_fname = mesh_path
if '.ply' in mesh_path.lower():
    print("Converting PLY to VTK...")
    mesh, vert_fname = convert_ply_to_vtk(mesh_path, save=True)

## Set up the SDF dataset
Sample SDFs across the surface using the same parameters as during model training (% near, far, and on surface).

In [ ]:
# Use fast mode or 2-phase optimization
fast_mode = True # @param {"type":"boolean"}

In [ ]:
# Prepare SDF dataset for shape completion
print("-----Preparing SDF Dataset-----")

if fast_mode==True:
    print("\033[92m-----Using Fast-Mode (Encoder + Refine)-----\033[0m")
    # Build the SDF dataset using downsampled surface samples
    points, sdf_vals, sdf_dataset, sample_dict = build_sdf_dataset(vert_fname, config, n_samples=None)

else:
    print("\033[92m-----Using 2-phase Optimization-----\033[0m")
    # Use a subset of the points for optizimation/reconstruction
    n_samples = 240 # @param {"type":"slider","min":100,"max":800,"step":20}

    # Build the SDF dataset using downsampled surface samples
    points, sdf_vals, sdf_dataset, sample_dict = build_sdf_dataset(vert_fname, config, n_samples)

# Save sampled points for downstream visualization
sdf_sample_pc = pv.PolyData(points.cpu().numpy())
sdf_sample_pc.save(os.path.join(outfpath, f"{mesh_name}_partial_input.vtk"))

# 4. Shape completion
---
Optimize partial point cloud latents and complete the shapes.

## Optimize latents
Since NSM is an auto-decoder framework, novel data cannot be encoded into the model the exact same was as training data.

To encode new data into the model's latent space, either optimization or a separately trained encoder is required, which is similar to a mini training session.

Choose "fast_mode" to use the PointNet encoder. Otherwise, 2-phase optimization will be used.

PointNet is 7.5x faster and more accurate for most modern vertebrae, but for fossils, performance varies.

Every fossil and dataset are different, and they can be adjusted to fit your use case. See [project Wiki](https://github.com/3D-fossils-Haag/nsm/wiki/Fossil-shape-completion:-Optimization-guide-for-hyperparameters) for more details on 2-phase optimization.

In [ ]:
# Encode latents using fast mode (encoder+refine) or 2-phase optimization

# Fast mode
if fast_mode:
    print("\n\033[33m-----Fast Mode (PointNet Encoder + Refinement): This will take ~1 minutes----\033[0m")
    encoder_path = MODEL_DIR + "/encoder/checkpoints/encoder.pt" # TO DO: Point to encoder ckpt
    encoder_ckpt = os.path.abspath(encoder_path)
    refine_iters = 200
    refine_lr = 1e-3
    refine_lambda_reg = 1e-6


    # Fast mode: one-shot encoder instead of full latent optimization
    print("\n-----Fast mode: encoding latent (single forward pass)----\n")
    latent_opt = encode_latent_pointnet(encoder_ckpt, points, sdf_vals, device)
    if refine_iters > 0:
            print(f"Refining encoder latent for {refine_iters} iters (lr={refine_lr}, lambda={refine_lambda_reg})...")
            latent_opt, _ = optimize_latent_partial(model, points.squeeze(), sdf_vals, config['latent_size'], latent_init=latent_opt, top_k=top_k_reg,
                                                    iters=refine_iters, lr=refine_lr, lambda_reg=refine_lambda_reg, clamp_val=None, latent_std=latent_std, scheduler_step=800,
                                                    scheduler_gamma=0.7, batch_inference_size=32768, multi_stage=True, device=device)

# 2-phase optimization
else:
    print("\n\033[33m-----2-phase Latent Optimization: This will take ~6 minutes----\033[0m")
    # Define best config for 2-phase optimization parameters
    best_cfg = {"top_k": 466,
                "iters1": 3000,
                "iters2": 8000,
                "lr1": 1e-4,
                "lr2": 1e-5,
                "lambda1": 1e-3,
                "lambda2": 1e-5,
                "clamp1": 1,
                "clamp2": None,
                "latent_std": torch.tensor(0.4401),
                "sched_step": 800,
                "sched_gamma1": 0.9,
                "sched_gamma2": 0.7,
                "batch_infer": 32768,
                "gridN": 256}

    # Encode latent via 2 stage optimization (auto-decoder framework)
    latent_opt = encode_latent(decoder=model, points=points.squeeze(), sdf_vals=sdf_vals, latent_dim=latent_codes.shape[1],
                                    mean_latent=mean_latent, latent_codes=latent_codes, top_k_reg=top_k_reg, latent_std=best_cfg['latent_std'],
                                    iters1=best_cfg['iters1'], iters2=best_cfg['iters2'], lr1=best_cfg['lr1'], lr2=best_cfg['lr2'],
                                    lambda_reg1=best_cfg['lambda1'], lambda_reg2=best_cfg['lambda2'], clamp_val1=best_cfg['clamp1'], clamp_val2=best_cfg['clamp2'],
                                    scheduler_step1=best_cfg['sched_step'], scheduler_step2=best_cfg['sched_step'], scheduler_gamma1=best_cfg['sched_gamma1'],
                                    scheduler_gamma2=best_cfg['sched_gamma2'], batch_inference_size=best_cfg['batch_infer'])

## Complete mesh from optimized latent
Feed optimized latent into trained model and reconstruct it into a completed mesh. Adjust n_pts_per_axis to increase or decrease the resolution of the produced mesh.

In [ ]:
# Reconstruction mesh from latent
mesh_out = reconstruct_mesh_from_latent(vert_fname, model, latent_opt, config)

# Normalize and scale output using model config training params
center, max_radius = get_norm_params(sdf_dataset, sample_dict, vert_fname)
mesh_pv = normalize_mesh(mesh_out, vert_fname, config, center, max_radius)

# Save mesh
output_path = os.path.join(outfpath, f"{mesh_name}_shape_completion.vtk")
color = np.array([112, 215, 222], dtype=np.uint8)  # RGB color
rgb = np.tile(color, (mesh_pv.n_points, 1))
mesh_pv.point_data.clear()
mesh_pv.point_data['Colors'] = rgb
mesh_pv.save(output_path)

print(f"\n\033[92mCompleted mesh saved to: {output_path}\033[0m")

# 5. Inspect Results
---
Plot the original versus shape completed mesh. Rerun and adjust optimization parameters as needed. The green points plotted on the completed mesh are the sampled points based on percentage near, far, and on the surface chosen in above section, "Prepare partial pointcloud for shape completion."

In [ ]:
#@title Plot the original mesh before preprocessing

# Check if used smoothed mesh or original mesh for inference
if "smooth" in mesh_name:
  smooth = True
  print("\n\033[92mSmoothed mesh used for inference.")
  mesh_name_orig = mesh_name.replace("_smooth_hollow", "")
  print(f"Visualizing original mesh before preprocessing from: {mesh_name_orig}.ply\033[0m\n")
else:
  smooth = False
  mesh_name_orig = mesh_name
  print("Original mesh used for inference")

# Read mesh
original_mesh = pv.read(os.path.join(mesh_dir, f"{mesh_name_orig}.ply"))
original_mesh.compute_normals(inplace=True)

# Plot figure
fig = go.Figure()
trace = pv_to_plotly(original_mesh, 'goldenrod', 1)
trace.name = "Original mesh"
fig.add_trace(trace)
for trace in fig.data:
    trace.showlegend = True
fig.update_layout(title=dict(text=f"Original Mesh (before completion)<br>{mesh_name_orig}",
                             x=0.5, y=0.95, xanchor="center", yanchor="top"),
                  showlegend=True,
                  scene_aspectmode='data',
                  legend=dict(x=1.02, y=1, bgcolor="rgba(255,255,255,0.7)",
                              bordercolor="black", borderwidth=1),
                  margin=dict(l=10, r=10, b=10, t=80))
fig.show()

In [ ]:
#@title Plot the original mesh with preprocessing - Smooth and hollow

if smooth:
  print("\n\033[92mSmoothed and hollowed mesh used for inference. Displaying preprocessed mesh before shape completion.\n\033[0m")
  # Read mesh
  original_mesh = pv.read(os.path.join(mesh_dir, f"{mesh_name}.vtk"))
  original_mesh.compute_normals(inplace=True)

  # Plot figure
  fig = go.Figure()
  trace = pv_to_plotly(original_mesh, 'goldenrod', 1)
  trace.name = "Original mesh"
  fig.add_trace(trace)
  for trace in fig.data:
      trace.showlegend = True
  fig.update_layout(title=dict(text=f"Original Mesh (before completion)<br>Smooth and Hollow<br>{mesh_name}",
                             x=0.5, y=0.95, xanchor="center", yanchor="top"),
                  showlegend=True,
                  scene_aspectmode='data',
                  legend=dict(x=1.02, y=1, bgcolor="rgba(255,255,255,0.7)",
                              bordercolor="black", borderwidth=1),
                  margin=dict(l=10, r=10, b=10, t=80))
  fig.show()

else:
  print("\n\033[92mOriginal mesh used for shape completion. No smooth/preprocessed mesh to show.\033[92m\n")

In [ ]:
#@title Plot the completed mesh vs sampled point cloud

print("\n\033[92mRendering shape completed mesh and point cloud used for SDF sampling.\033[0m\n")

show_SDF_samples = False # @param {"type":"boolean"}
sdf_sample_pt_sz = 1 # @param {"type":"slider","min":1,"max":5,"step":1}

# Read mesh
sdf_sample_pc = pv.read(os.path.join(outfpath, f"{mesh_name}_partial_input.vtk"))
completed_mesh = pv.read(output_path)

# Plot figure
fig = go.Figure()
trace = (pv_to_plotly(completed_mesh, 'deepskyblue', 1)) # Completed surface
trace.name = "Completed mesh"
if show_SDF_samples:
    fig.add_trace(trace)
    trace = (pv_points_to_plotly(sdf_sample_pc, 'darkseagreen', size=sdf_sample_pt_sz)) # SDF sampled points
    trace.name = "SDF samples"
fig.add_trace(trace)
for trace in fig.data:
    trace.showlegend = True
fig.update_layout(title=dict(text=f"Completed mesh<br>{mesh_name}",
                             x=0.5, y=0.95, xanchor="center", yanchor="top"),
                  showlegend=True,
                  scene_aspectmode='data',
                  legend=dict(x=1.02, y=1, bgcolor="rgba(255,255,255,0.7)",
                              bordercolor="black", borderwidth=1),
                  margin=dict(l=10, r=10, b=10, t=80))
fig.show()

# Optional: Visualize uncertainty of shape completion results on mesh (experimental)
---
This is a new tool under development.    
1) Use Laplace approximation to find the covariance threshold for a provided latent code before performance worsens.   
2) Use the determined covariance threshold to create many new slightly different latent vectors and calculate shape completion variance per point.    
3) Project SDF variance values on surface.


In [ ]:
#@title Estimate uncertainty
import NSM.uncertainty as uncert

# Laplace approximator
laplace = uncert.LaplaceApproximator(model, latent_opt, mesh_path, config, device, verbose=True)

# Uncertainty propagator
propagation_mode = "montecarlo" # @param ["montecarlo","analytical"]
if propagation_mode == 'analytical':
    propagator = uncert.AnalyticalUncertainty(model, latent_opt, device, verbose=True)
elif propagation_mode == 'montecarlo':
    propagator = uncert.MonteCarloUncertainty(model, latent_opt, device, verbose=True)

# TO DO: Define parameters
# Expected data std dev (noise), based on quality of input mesh
data_std = 2e-5 #@param

# Expected latent std dev (noise), based on stability of optimal latent
latent_std = 5e-4 #@param

# Weight of reconstruction (data) term
data_weight = 1 #@param

# Weight of regularization (latent) term
latent_weight = 1 #@param

# Compute approximate covariance of latent posterior
covariance, hessian = laplace.covariance(data_std, latent_std, data_weight, latent_weight, n_samples=2000, recompute_jacobian=False)

# Compute SDF uncertainty at mesh vertices
recon_mesh_dec = uncert.decimate_mesh(completed_mesh, n_triangles=5000, verbose=True) # Reduce vertex count to speed up computation
if propagation_mode == 'analytical':
    surface_sdf_std = propagator.sdf_uncertainty(covariance, recon_mesh_dec.points, recompute_gradients=False)
elif propagation_mode == 'montecarlo':
    surface_sdf_std = propagator.sdf_uncertainty(covariance, recon_mesh_dec.points, n_samples=2000)

# Update mesh point data and save
# Create output directory for current hyperparameters
OUTPUT_UNC_DIR = os.path.join(outfpath, f"data{data_std if data_weight else 'inf'}_lat{latent_std if latent_weight else 'inf'}")
os.makedirs(OUTPUT_UNC_DIR, exist_ok=True)

# Update mesh data
recon_mesh_dec.point_data['SdfUncertainty'] = surface_sdf_std.cpu().numpy()
recon_mesh_dec_unc_path = os.path.join(OUTPUT_UNC_DIR, f"{mesh_name}_shape_completion_unc.vtk")
recon_mesh_dec.save(recon_mesh_dec_unc_path)
print("Reconstructed mesh with SDF uncertainty saved to:")
print(recon_mesh_dec_unc_path)

# Prepare mesh variables for plotting
uncert_mesh = uncert.unit_scale_mesh(recon_mesh_dec).copy()
uncert_mesh["SdfUncertainty"] = recon_mesh_dec["SdfUncertainty"]
scalar_name = 'SdfUncertainty'

In [ ]:
#@title Plot mesh with uncertainty
print("\n\033[92mRendering shape completed mesh with SDF uncertainty.\033[0m\n")

# Plot figure
fig = go.Figure()
trace = pv_to_plotly(uncert.unit_scale_mesh(recon_mesh_dec), scalars=recon_mesh_dec[scalar_name])
trace.name = "Reconstructed mesh (uncertainty)"
trace.colorbar = dict(title=scalar_name, titlefont=dict(family='times'), tickfont=dict(family='times'))
fig.add_trace(trace)
for trace in fig.data:
    trace.showlegend = True
fig.update_layout(title=dict(text=f"Reconstructed Mesh with SDF Uncertainty<br>{mesh_name}",
                             x=0.5, y=0.95, xanchor="center", yanchor="top"),
                  showlegend=True, scene_aspectmode='data',
                  legend=dict(x=1.02, y=1, bgcolor="rgba(255,255,255,0.7)",
                              bordercolor="black", borderwidth=1),
                  margin=dict(l=10, r=10, b=10, t=80), scene=dict(
                      xaxis=dict(visible=False), yaxis=dict(visible=False),
                      zaxis=dict(visible=False), bgcolor="white"))
fig.show()

# Optional: Save a spinning gif of your 3D models
---

In [ ]:
from google.colab import files

# Read mesh
mesh_type = "uncertainty" # @param ["original","completed","uncertainty"] {"allow-input":true}
if mesh_type == "completed":
    mesh = completed_mesh
elif mesh_type == "uncertainty":
    mesh = uncert_mesh
else:
    mesh = original_mesh

# Create plotter
plotter = pv.Plotter(off_screen=True, window_size=(600, 600))
plotter.set_background("white")
if "uncertainty" not in mesh_type:
    plotter.add_mesh(mesh, color="deepskyblue", opacity=1.0)
else:
    plotter.add_mesh(mesh, scalars="SdfUncertainty",
                     cmap="viridis", show_scalar_bar=True)

# Open GIF
gif_path = f"/tmp/{mesh_name}_{mesh_type}_spin.gif"
plotter.open_gif(gif_path, fps=30)

# Camera setup
plotter.camera.zoom(1.2)

# Rotate + write frames
n_frames = 120 # @param {"type":"slider","min":60,"max":360,"step":60}
for i in range(n_frames):
    if i % 15 == 0:
        print(f"Frame {i}/{n_frames}")
    plotter.camera.Azimuth(360 / n_frames)
    plotter.write_frame()
plotter.close()

print(f"Saved GIF to {wd}\n{gif_path}")

# Download to your local machine
files.download(gif_path)